# 逐事件軌跡動畫(replay_epoch)

`example/replay_epoch.py` 用某個 epoch 存的權重、強制 `chunk_size=1` 重跑
`run_network_traced`,拿到逐事件精確的 `LayerForwardTrace`(`spike_mask`/
`v_steps`/`event_ms`,3 欄,見該檔案 docstring)。這份 notebook 把它依
`event_ms` 攤成真實時間軸播放成動畫:離散量(`spike_mask`)用瞬時顯示
(`viz.time_resample.resample_pulse`——spike 是瞬時事件,事件跟事件之間這個
neuron 根本沒被計算過,不是「還在 fire」,所以沒有事件的 frame 一律顯示沒
fire,不會沿用前一筆事件的值),連續量(`v_steps`)用衰減插值(`resample_decay`,
用該層的 `tau`,電位是真的連續存在的物理量,沿用前一筆事件的值再衰減才有
意義)——插值/動畫繪圖邏輯都在 `viz/`,這裡只負責讀資料、組裝、擺 UI。

conv 層跟 FC 層呈現方式不一樣(conv 攤成 `(oc,h,w)` 圖排網格;FC 沒有空間
結構,改成選 neuron 範圍 x 滑動時間窗口),各自「怎麼呈現自己」收進
`example/replay_panels.py` 的 `ConvChannelPanel`/`FCWindowPanel`——這裡只負責
組裝、擺 UI,不重複寫換算邏輯。

In [1]:
%matplotlib inline
import math

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import HTML, display

from example.paths import EXPERIMENTS_DIR
from example.replay_epoch import _load_train_sample, load_epoch_weights, replay_sample
from example.replay_panels import ConvChannelPanel, FCWindowPanel
from example.utils import load_run_record
from salt_core.layers import ConvLayer, FCLayer
from viz.channel_grid import ChannelGridAnimation
from viz.time_resample import build_frame_grid


## 參數

改這裡,不改函式庫。

In [2]:
RUN_NAME = "conv_compressed_compressed_maxsteps_verify_20260913_120425"
EXP_DIR = str(EXPERIMENTS_DIR / RUN_NAME)
EPOCH = 0            # 要有對應的 weights/epoch_XXX.npz
DT_MS = 1.0          # 動畫的時間格解析度(毫秒),越小動畫越平滑但 frame 數越多
N_CONV_PANELS = 2    # 要幾個 conv 窗口
N_FC_PANELS = 2      # 要幾個 FC 窗口
NCOLS = 2            # conv 窗口排成幾欄(FC 窗口固定另外上下並排,見下面組裝)
FC_SIZE_RATIO = (2.0, 0.5)   # FC 窗口的(寬倍率, 高倍率)——拉寬、壓扁


## 讀這個 epoch 的層(幾何/`tau` 只跟 epoch 有關,跟 sample 無關,只需要讀一次)

conv 層跟 FC 層分開放進兩份選單,因為呈現方式完全不同(conv 攤成 `(oc,h,w)`
圖排網格;FC 沒有空間結構,見下面「FC 層」那節)。

In [3]:
LAYERS, _ = load_epoch_weights(EXP_DIR, EPOCH)
RUN_RECORD = load_run_record(EXP_DIR)

CONV_LAYERS = [l for l in LAYERS if isinstance(l, ConvLayer)]
CONV_LAYER_NAMES = [l.name for l in CONV_LAYERS]
print("conv 層:", CONV_LAYER_NAMES)

FC_LAYERS = [l for l in LAYERS if isinstance(l, FCLayer)]
FC_LAYER_NAMES = [l.name for l in FC_LAYERS]
print("FC 層:", FC_LAYER_NAMES, [l.n_neurons for l in FC_LAYERS])


conv 層: ['conv1', 'conv2']
FC 層: ['out'] [10]


## 組裝:conv 跟 FC 一起比較

先在上面「參數」設定要幾個 conv 窗口、幾個 FC 窗口(`N_CONV_PANELS`/
`N_FC_PANELS`)。每一格獨立選自己的設定——conv 選 `sample`/`layer`/
`quantity`/`channel`;FC 選 `sample`/`layer`/`quantity`/neuron 範圍/窗口
寬度。全部格子共用同一條真實 ms 時間軸,按一次「畫」把所有格子(不管幾個
conv、幾個 FC)組成同一組動畫,一個 Play 鍵同步播放。

畫面排法固定是 `Conv Conv / FC / FC`:conv 窗口照 `NCOLS` 自動排版在最上面;
FC 窗口排在 conv 下面、彼此上下並排,而且**橫跨全部 conv 欄位**(不是塞進
單一欄)——GridSpec 同一欄所有列要共用同一個寬度,FC 如果只佔其中一欄、又
要求變寬,同欄的 conv 會被迫跟著變寬,conv 之間就會被拉開一大段空白(實測
踩過這個坑)。橫跨全部欄位就不會有這個問題:FC 要的寬度直接由「conv 全部
欄位的寬度總和」提供,不逼任何一欄單獨變寬。

`ConvChannelPanel`/`FCWindowPanel`(見 `example/replay_panels.py`)建好之後,
在這裡直接設 `panel.position`/`panel.size_ratio` 兩個屬性蓋掉自動排版
(`position` 的欄位可以是單一整數,也可以是 `slice` 橫跨好幾欄),`viz/`
那邊完全不知道 conv/FC 是什麼,只讀這兩個屬性排版。不同形狀的格子(conv
接近正方形、FC 窗口又寬又扁)各自維持自己該有的長寬比例,不會被拉伸。

換 `sample`/`epoch` 才需要重跑 forward,所以 traces 用 `(epoch, sample)` 當
key cache 起來——只留最新一筆,不無限累積記憶體。

In [ ]:
_traces_cache = {}


def _get_traces(sample: int):
    key = (EPOCH, sample)
    if key not in _traces_cache:
        _traces_cache.clear()
        raw_sample = _load_train_sample(RUN_RECORD, sample)
        _traces_cache[key] = replay_sample(EXP_DIR, EPOCH, *raw_sample)
    return _traces_cache[key]


def _trace_for(sample: int, layer_name: str):
    idx = next(i for i, l in enumerate(LAYERS) if l.name == layer_name)
    return _get_traces(sample)[idx]


def _conv_row(default_channel: int = 0):
    return widgets.HBox([
        widgets.IntText(value=0, description="sample", layout=widgets.Layout(width="150px")),
        widgets.Dropdown(options=CONV_LAYER_NAMES, description="layer", layout=widgets.Layout(width="190px")),
        widgets.Dropdown(options=["spike_mask", "v_steps"], description="quantity",
                          layout=widgets.Layout(width="220px")),
        widgets.IntText(value=default_channel, description="channel", layout=widgets.Layout(width="150px")),
    ])


def _fc_row(default_hi: int = 10):
    return widgets.HBox([
        widgets.IntText(value=0, description="sample", layout=widgets.Layout(width="150px")),
        widgets.Dropdown(options=FC_LAYER_NAMES, description="layer", layout=widgets.Layout(width="190px")),
        widgets.Dropdown(options=["spike_mask", "v_steps"], description="quantity",
                          layout=widgets.Layout(width="220px")),
        widgets.IntText(value=0, description="neuron lo", layout=widgets.Layout(width="150px")),
        widgets.IntText(value=default_hi, description="neuron hi", layout=widgets.Layout(width="150px")),
        widgets.FloatText(value=10.0, description="window±ms", layout=widgets.Layout(width="160px")),
    ])


_conv_rows = [_conv_row() for _ in range(N_CONV_PANELS)]
_fc_default_hi = min(10, FC_LAYERS[0].n_neurons) if FC_LAYERS else 10
_fc_rows = [_fc_row(_fc_default_hi) for _ in range(N_FC_PANELS)]
_go_button = widgets.Button(description="畫")
_go_output = widgets.Output()


def _build_panels_and_frame_ms():
    conv_specs = [tuple(child.value for child in row.children) for row in _conv_rows]
    fc_specs = [tuple(child.value for child in row.children) for row in _fc_rows]

    t_end = 0.0
    for sample, layer_name, _, _ in conv_specs:
        t_end = max(t_end, float(np.nanmax(np.asarray(_trace_for(sample, layer_name).event_ms))))
    for sample, layer_name, _, _, _, _ in fc_specs:
        t_end = max(t_end, float(np.nanmax(np.asarray(_trace_for(sample, layer_name).event_ms))))
    frame_ms = build_frame_grid(0.0, t_end, DT_MS)

    panels = []
    for sample, layer_name, quantity, channel in conv_specs:
        layer = next(l for l in CONV_LAYERS if l.name == layer_name)
        trace = _trace_for(sample, layer_name)
        panels.append(ConvChannelPanel(layer, channel, quantity, trace, frame_ms, DT_MS,
                                        label_prefix=f"s{sample} / "))

    # FC 窗口排在 conv 下面、彼此上下並排,而且橫跨全部 conv 欄位(不是塞進
    # 單一欄)——蓋掉 viz/channel_grid.py 的自動排版。橫跨全部欄位才不會逼
    # 任何一欄單獨變寬:FC 要的寬度由「conv 全部欄位的寬度總和」提供,見上面
    # 說明 cell。
    conv_nrows = math.ceil(len(conv_specs) / NCOLS) if conv_specs else 0
    for i, (sample, layer_name, quantity, neuron_lo, neuron_hi, window_ms) in enumerate(fc_specs):
        layer = next(l for l in FC_LAYERS if l.name == layer_name)
        trace = _trace_for(sample, layer_name)
        panel = FCWindowPanel(layer, neuron_lo, neuron_hi, quantity, window_ms,
                               trace, frame_ms, DT_MS, label_prefix=f"s{sample} / ")
        panel.position = (conv_nrows + i, slice(0, NCOLS))
        panel.size_ratio = FC_SIZE_RATIO
        panels.append(panel)
    return panels, frame_ms


def _on_go_clicked(_):
    with _go_output:
        _go_output.clear_output(wait=True)
        panels, frame_ms = _build_panels_and_frame_ms()
        anim = ChannelGridAnimation(ncols=NCOLS).build(
            panels, frame_labels=[f"t = {ms:.1f} ms" for ms in frame_ms])
        plt.close(anim._fig)  # 避免 %matplotlib inline 把靜態圖跟 jshtml 動畫重複顯示
        display(HTML(anim.to_jshtml()))


_go_button.on_click(_on_go_clicked)
display(widgets.VBox(
    [widgets.HTML("<b>conv 窗口</b>")] + _conv_rows +
    [widgets.HTML("<b>FC 窗口</b>")] + _fc_rows +
    [_go_button, _go_output]
))
